In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
E = 1

In [5]:
x_train = np.load('Train_3D_128_FOV_norm_data.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))
vol_shape = x_train.shape[1:]
print('train vol_shape:', vol_shape)
print('train shape:', x_train.shape)

train vol_shape: (128, 128, 128)
train shape: (57, 128, 128, 128)


In [6]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [7]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 128, 128])
Fixed Images Shape: torch.Size([2, 1, 128, 128, 128])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 128, 128])
Zero Gradient Shape: (2, 128, 128, 128, 3)


In [8]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    return mse

In [9]:
def load_partial_state_dict(model, state_dict):
    model_dict = model.state_dict()
    pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict and model_dict[k].size() == v.size()}
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

In [10]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64],
    [64, 64, 64, 64, 32, 16, 16]
]

In [11]:
model3D_1 = vxm.networks.VxmDense((16, 16, 16), nb_features, int_steps=0)
model3D_1.to(device)
optimizer = optim.Adam(model3D_1.parameters(), lr=1e-4)

5
6
7
6
7
6
7
6
7
8
9
10
8
9
10
8
9
10
8
9
10
11
11
11
5
6
7
6
7
6
7
6
7
8
9
10
8
9
10
8
9
10
8
9
10
11
11
11
[8, 8, 8]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [12]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images = torch.nn.functional.interpolate(moving_images, size=(16,16,16), mode='trilinear', align_corners=False)
    fixed_images = torch.nn.functional.interpolate(fixed_images, size=(16,16,16), mode='trilinear', align_corners=False)
    
    moving_images = moving_images.to(device)
    fixed_images = fixed_images.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, _ = model3D_1(moving_images, fixed_images)
    loss, mse, grad = total_loss(fixed_images, transformed_image)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_1.state_dict(), 'model_VXM_3D_weights_V3_1_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    mses.append(mse.cpu().item())
    grads.append(grad.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Mses: {mses}, Grad: {grad}")

  0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\user\AppData\Local\Temp\ipykernel_26224\3825517738.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
C:\Users\user\AppData\Local\Temp\ipykernel_26224\3825517738.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)


ValueError: too many values to unpack (expected 2)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')
plt.plot(epochs_range, mses, label='MSE')
plt.plot(epochs_range, grads, label='Gradient')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_2 = vxm.networks.VxmDense((32, 32, 32), nb_features, int_steps=0)
model3D_2.to(device)
optimizer = optim.Adam(model3D_2.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

partial_weights = torch.load('model_VXM_3D_weights_V3_1_omomi.pth', map_location=device)
load_partial_state_dict(model3D_2, partial_weights)

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images_16 = torch.nn.functional.interpolate(moving_images, size=(16,16,16), mode='trilinear', align_corners=False)
    fixed_images_16 = torch.nn.functional.interpolate(fixed_images, size=(16,16,16), mode='trilinear', align_corners=False)    
    moving_images_16 = moving_images_16.to(device)
    fixed_images_16 = fixed_images_16.to(device)
    
    moving_images_32 = torch.nn.functional.interpolate(moving_images, size=(32,32,32), mode='trilinear', align_corners=False)
    fixed_images_32 = torch.nn.functional.interpolate(fixed_images, size=(32,32,32), mode='trilinear', align_corners=False)    
    moving_images_32 = moving_images_32.to(device)
    fixed_images_32 = fixed_images_32.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image_16, Vector_16 = model3D_1(moving_images_16, fixed_images_16)
    transformed_image_32, Vector_32 = model3D_2(moving_images_32, fixed_images_32)
    
    Vector_16_32 = torch.nn.functional.interpolate(Vector_16, size=(32,32,32), mode='trilinear', align_corners=False)
    Vector_16_32 = Vector_16_32*2
  
    loss = MSE_Loss(Vector_16_32, Vector_32)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_2.state_dict(), 'model_VXM_3D_weights_V3_2_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_3 = vxm.networks.VxmDense((32, 32, 32), nb_features, int_steps=0)
model3D_3.to(device)
optimizer = optim.Adam(model3D_3.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

model3D_3.load_state_dict(torch.load('model_VXM_3D_weights_V3_2_omomi.pth', map_location=device))

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images = torch.nn.functional.interpolate(moving_images, size=(32,32,32), mode='trilinear', align_corners=False)
    fixed_images = torch.nn.functional.interpolate(fixed_images, size=(32,32,32), mode='trilinear', align_corners=False)
    
    moving_images = moving_images.to(device)
    fixed_images = fixed_images.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, _ = model3D_3(moving_images, fixed_images)
    loss, mse, grad = total_loss(fixed_images, transformed_image)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_3.state_dict(), 'model_VXM_3D_weights_V3_3_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    mses.append(mse.cpu().item())
    grads.append(grad.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Mses: {mses}, Grad: {grad}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')
plt.plot(epochs_range, mses, label='MSE')
plt.plot(epochs_range, grads, label='Gradient')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_4 = vxm.networks.VxmDense((64, 64, 64), nb_features, int_steps=0)
model3D_4.to(device)
optimizer = optim.Adam(model3D_4.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

partial_weights = torch.load('model_VXM_3D_weights_V3_3_omomi.pth', map_location=device)
load_partial_state_dict(model3D_4, partial_weights)


for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images_32 = torch.nn.functional.interpolate(moving_images, size=(32,32,32), mode='trilinear', align_corners=False)
    fixed_images_32 = torch.nn.functional.interpolate(fixed_images, size=(32,32,32), mode='trilinear', align_corners=False)    
    moving_images_32 = moving_images_32.to(device)
    fixed_images_32 = fixed_images_32.to(device)
    
    moving_images_64 = torch.nn.functional.interpolate(moving_images, size=(64,64,64), mode='trilinear', align_corners=False)
    fixed_images_64 = torch.nn.functional.interpolate(fixed_images, size=(64,64,64), mode='trilinear', align_corners=False)    
    moving_images_64 = moving_images_64.to(device)
    fixed_images_64 = fixed_images_64.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image_32, Vector_32 = model3D_3(moving_images_32, fixed_images_32)
    transformed_image_64, Vector_64 = model3D_4(moving_images_64, fixed_images_64)
    
    Vector_32_64 = torch.nn.functional.interpolate(Vector_32, size=(64,64,64), mode='trilinear', align_corners=False)
    Vector_32_64 = Vector_32_64*2
  
    loss = MSE_Loss(Vector_32_64, Vector_64)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_4.state_dict(), 'model_VXM_3D_weights_V3_4_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_5 = vxm.networks.VxmDense((64, 64, 64), nb_features, int_steps=0)
model3D_5.to(device)
optimizer = optim.Adam(model3D_5.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

model3D_5.load_state_dict(torch.load('model_VXM_3D_weights_V3_4_omomi.pth', map_location=device))

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images = torch.nn.functional.interpolate(moving_images, size=(64,64,64), mode='trilinear', align_corners=False)
    fixed_images = torch.nn.functional.interpolate(fixed_images, size=(64,64,64), mode='trilinear', align_corners=False)
    
    moving_images = moving_images.to(device)
    fixed_images = fixed_images.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, _ = model3D_5(moving_images, fixed_images)
    loss, mse, grad = total_loss(fixed_images, transformed_image)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_5.state_dict(), 'model_VXM_3D_weights_V3_5_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    mses.append(mse.cpu().item())
    grads.append(grad.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Mses: {mses}, Grad: {grad}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')
plt.plot(epochs_range, mses, label='MSE')
plt.plot(epochs_range, grads, label='Gradient')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_6 = vxm.networks.VxmDense((128, 128, 128), nb_features, int_steps=0)
model3D_6.to(device)
optimizer = optim.Adam(model3D_6.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

model3D_6.load_state_dict(torch.load('model_VXM_3D_weights_V3_5_omomi.pth', map_location=device))

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images_64 = torch.nn.functional.interpolate(moving_images, size=(64,64,64), mode='trilinear', align_corners=False)
    fixed_images_64 = torch.nn.functional.interpolate(fixed_images, size=(64,64,64), mode='trilinear', align_corners=False)    
    moving_images_64 = moving_images_64.to(device)
    fixed_images_64 = fixed_images_64.to(device)
        
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image_64, Vector_64 = model3D_5(moving_images_64, fixed_images_64)
    transformed_image_128, Vector_128 = model3D_6(moving_images, fixed_images)
    
    Vector_64_128 = torch.nn.functional.interpolate(Vector_64, size=(128,128,128), mode='trilinear', align_corners=False)
    Vector_64_128 = Vector_64_128*2
  
    loss = MSE_Loss(Vector_64_128, Vector_128)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_6.state_dict(), 'model_VXM_3D_weights_V3_6_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()

In [ ]:
model3D_7 = vxm.networks.VxmDense((128, 128, 128), nb_features, int_steps=0)
model3D_7.to(device)
optimizer = optim.Adam(model3D_7.parameters(), lr=1e-4)

In [ ]:
from tqdm.notebook import tqdm

epochs = E
losses = []
mses = []
grads = []

partial_weights = torch.load('model_VXM_3D_weights_V3_6_omomi.pth', map_location=device)
load_partial_state_dict(model3D_7, partial_weights)

for epoch in tqdm(range(epochs)):
    
    # 学習データのバッチを取得
    train_batch,_ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32)
    fixed_images = torch.tensor(train_batch[1], dtype=torch.float32)
    
    moving_images = moving_images.to(device)
    fixed_images = fixed_images.to(device)
    # 勾配を初期化
    optimizer.zero_grad()

    # 順伝播
    transformed_image, _ = model3D_7(moving_images, fixed_images)
    loss, mse, grad = total_loss(fixed_images, transformed_image)
    
    # 逆伝播
    loss.backward()
    optimizer.step()

    torch.save(model3D_7.state_dict(), 'model_VXM_3D_weights_V3_7_omomi.pth')

        
    # エポックごとのロスを保存

    
    losses.append(loss.cpu().item())
    mses.append(mse.cpu().item())
    grads.append(grad.cpu().item())
    

    # エポックごとのロスの表示
#     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_all:.4f}")
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Mses: {mses}, Grad: {grad}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 損失、MSE、勾配をプロット
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 8))
plt.plot(epochs_range, losses, label='Loss')
plt.plot(epochs_range, mses, label='MSE')
plt.plot(epochs_range, grads, label='Gradient')

plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Training Metrics')
plt.legend()
plt.show()